In [ ]:
# Cell 1: Install required libraries   
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cpu.html -q
!pip install captum -q
!pip install shap -q
!pip install pygcl -q

print("✅ All libraries installed")

In [ ]:
# ============================================================
# CELL 1: Cài đặt và Import (CHẠY SAU KHI RESTART KERNEL)
# ============================================================
import os
import gc
import sys
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import deque

warnings.filterwarnings('ignore')

# Set seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# Import PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Import sklearn
from sklearn.preprocessing import RobustScaler, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, 
    confusion_matrix, precision_recall_fscore_support, 
    roc_auc_score, roc_curve
)

# Import PyTorch Geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.utils import dropout_edge, to_undirected

set_seed(42)
print("✅ All imports completed")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ============================================================
# CELL 2: DWB Loss (FIXED - Không bị NaN)
# ============================================================
class DWBLoss(nn.Module):
    """
    Dynamic Weighted Balanced Loss - Ổn định, không NaN
    """
    def __init__(self, num_classes, class_counts=None, gamma=2.0, use_brier=True):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma
        self.use_brier = use_brier
        
        if class_counts is not None:
            class_counts = np.array(class_counts, dtype=np.float32)
            class_counts = class_counts / (class_counts.sum() + 1e-8)
            self.register_buffer('class_counts', torch.tensor(class_counts, dtype=torch.float))
        else:
            self.register_buffer('class_counts', torch.ones(num_classes) / num_classes)
    
    def forward(self, logits, targets):
        # Clamp logits để tránh overflow
        logits = torch.clamp(logits, min=-30, max=30)
        
        # Tính probs an toàn
        probs = F.softmax(logits, dim=-1)
        probs = torch.clamp(probs, min=1e-7, max=1.0 - 1e-7)
        
        p_ij = probs.gather(1, targets.unsqueeze(-1)).squeeze(-1)
        p_ij = torch.clamp(p_ij, min=1e-7, max=1.0 - 1e-7)
        
        # Trọng số động
        w_j = 1.0 / (self.class_counts + 1e-7)
        w_j = w_j / (w_j.sum() + 1e-7) * self.num_classes
        w_j = torch.clamp(w_j, min=0.1, max=10.0)
        w_sample = w_j[targets]
        
        # Dynamic weight
        dynamic_weight = w_sample ** (1 - p_ij)
        dynamic_weight = torch.clamp(dynamic_weight, min=0.1, max=10.0)
        
        # Cross-entropy
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        weighted_ce = dynamic_weight * ce_loss
        
        if self.use_brier:
            brier_term = -p_ij * (1 - p_ij)
            loss = (weighted_ce + 0.1 * brier_term).mean()
        else:
            loss = weighted_ce.mean()
        
        # Kiểm tra NaN và fallback
        if torch.isnan(loss) or torch.isinf(loss):
            loss = F.cross_entropy(logits, targets)
        
        return loss


class DWBWithLogitAdjustment(DWBLoss):
    def __init__(self, num_classes, class_counts=None, gamma=2.0, use_brier=True, tau=0.05):
        super().__init__(num_classes, class_counts, gamma, use_brier)
        self.tau = tau
    
    def forward(self, logits, targets):
        with torch.no_grad():
            pi = self.class_counts / (self.class_counts.sum() + 1e-8)
            logit_adj = self.tau * torch.log(pi + 1e-8)
            logit_adj = torch.clamp(logit_adj, min=-5, max=5)
        
        logits_adjusted = logits + logit_adj.unsqueeze(0)
        return super().forward(logits_adjusted, targets)

print("✅ DWBLoss defined (stable)")

In [ ]:
# ============================================================
# CELL 3: Temporal GAT (Đơn giản hóa, ổn định)
# ============================================================
class TemporalGATConv(nn.Module):
    """
    Time-aware Graph Attention - Đơn giản và ổn định
    """
    def __init__(self, in_channels, out_channels, time_dim=8, heads=2, dropout=0.1):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        # Time encoding đơn giản
        self.time_encoder = nn.Linear(1, time_dim)
        
        # Projection
        self.proj = nn.Linear(in_channels + time_dim, out_channels)
        
        # GAT convolution thay vì MultiheadAttention (ổn định hơn)
        self.gat = GATConv(out_channels, out_channels, heads=heads, concat=False, dropout=dropout)
        
        self.norm = nn.LayerNorm(out_channels)
        self.dropout = nn.Dropout(dropout)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x, edge_index, edge_times, edge_attr=None):
        N = x.size(0)
        device = x.device
        
        # Encode time
        if edge_times is not None:
            edge_times = torch.clamp(edge_times, 0, 1)
            time_enc = self.time_encoder(edge_times.unsqueeze(-1))
        else:
            time_enc = torch.zeros(edge_index.size(1), self.time_encoder.out_features, device=device)
        
        # Node time embedding (average of incident edges)
        node_time = torch.zeros(N, self.time_encoder.out_features, device=device)
        for i in range(N):
            incident = (edge_index[0] == i) | (edge_index[1] == i)
            if incident.sum() > 0:
                node_time[i] = time_enc[incident].mean(dim=0)
        
        # Concatenate
        x_aug = torch.cat([x, node_time], dim=-1)
        x_proj = self.proj(x_aug)
        x_proj = F.gelu(x_proj)
        
        # GAT propagation
        out = self.gat(x_proj, edge_index)
        out = self.norm(out + x_proj)  # Residual
        out = self.dropout(F.gelu(out))
        
        return out

print("✅ TemporalGATConv defined (stable)")

In [ ]:
# ============================================================
# CELL 4: Improved AdaGraph (FIXED)
# ============================================================
class ImprovedAdaGraph(nn.Module):
    """
    AdaGraph với temporal modeling - Ổn định
    """
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64,
                 num_classes=2, class_counts=None, time_dim=8, dropout=0.1):
        super().__init__()
        
        self.node_proj = nn.Linear(node_in_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_in_dim, hidden_dim)
        
        nn.init.xavier_uniform_(self.node_proj.weight)
        nn.init.xavier_uniform_(self.edge_proj.weight)
        
        # Temporal layers
        self.temporal_conv1 = TemporalGATConv(hidden_dim, hidden_dim, time_dim, dropout=dropout)
        self.temporal_conv2 = TemporalGATConv(hidden_dim, hidden_dim, time_dim, dropout=dropout)
        
        if class_counts is None:
            class_counts = [1, 1]
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        
        self.loss_fn = DWBWithLogitAdjustment(num_classes, class_counts)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)
        
    def forward(self, graph, labels=None, return_loss=False):
        x = self.node_proj(graph.x)
        x = self.norm(x)
        x = F.gelu(x)
        
        edge_attr = self.edge_proj(graph.edge_attr)
        edge_attr = F.gelu(edge_attr)
        
        # Temporal
        if hasattr(graph, 'edge_time') and graph.edge_time is not None:
            edge_times = graph.edge_time
        else:
            edge_times = torch.arange(graph.edge_index.size(1), device=x.device).float()
            edge_times = edge_times / (edge_times.max() + 1)
        
        # Residual connections
        x1 = self.temporal_conv1(x, graph.edge_index, edge_times, edge_attr)
        x = x + x1
        x = F.gelu(x)
        x = self.dropout(x)
        
        x2 = self.temporal_conv2(x, graph.edge_index, edge_times, edge_attr)
        x = x + x2
        x = F.gelu(x)
        x = self.dropout(x)
        
        # Edge-level prediction
        src_emb = x[graph.edge_index[0]]
        dst_emb = x[graph.edge_index[1]]
        edge_repr = torch.cat([src_emb, dst_emb], dim=-1)
        
        logits = self.classifier(edge_repr)
        logits = torch.clamp(logits, min=-30, max=30)
        
        if return_loss and labels is not None:
            loss = self.loss_fn(logits, labels)
            return logits, loss
        return logits

print("✅ ImprovedAdaGraph defined (stable)")

In [ ]:
# ============================================================
# CELL 5: Generic Graph Builder (FIXED - Xử lý dữ liệu an toàn)
# ============================================================
class GenericGraphBuilder:
    """Universal graph builder - Xử lý dữ liệu an toàn"""
    
    def __init__(self, df, src_col=None, dst_col=None, label_col=None, 
                 feature_cols=None, node_id_col=None, time_col=None):
        self.df = df.copy()
        self.src_col = src_col
        self.dst_col = dst_col
        self.label_col = label_col
        self.feature_cols = feature_cols
        self.node_id_col = node_id_col
        self.time_col = time_col
        
        if self.src_col is None or self.dst_col is None:
            self._auto_detect_node_columns()
        if self.label_col is None:
            self._auto_detect_label_column()
        if self.feature_cols is None:
            self._auto_detect_feature_columns()
        if self.time_col is None:
            self._auto_detect_time_column()
    
    def _auto_detect_node_columns(self):
        node_candidates = []
        for col in self.df.columns:
            col_lower = col.lower()
            if 'src' in col_lower or 'source' in col_lower or 'from' in col_lower:
                node_candidates.append(('src', col))
            if 'dst' in col_lower or 'dest' in col_lower or 'target' in col_lower or 'to' in col_lower:
                node_candidates.append(('dst', col))
        
        if len(node_candidates) == 0:
            cols = self.df.columns.tolist()
            self.src_col = cols[0]
            self.dst_col = cols[1] if len(cols) > 1 else cols[0]
        else:
            for role, col in node_candidates:
                if role == 'src':
                    self.src_col = col
                elif role == 'dst':
                    self.dst_col = col
        
        print(f"📌 Source: '{self.src_col}', Destination: '{self.dst_col}'")
    
    def _auto_detect_label_column(self):
        common_labels = ['label', 'Label', 'class', 'Class', 'attack', 'Attack',
                         'type', 'Type', 'category', 'Category', 'target']
        for col in common_labels:
            if col in self.df.columns:
                self.label_col = col
                print(f"📌 Label: '{self.label_col}'")
                return
        
        for col in self.df.columns:
            if self.df[col].nunique() <= 10:
                self.label_col = col
                print(f"📌 Label (auto): '{self.label_col}'")
                return
        
        self.label_col = None
    
    def _auto_detect_feature_columns(self):
        exclude_cols = [self.src_col, self.dst_col, self.label_col, self.node_id_col, self.time_col]
        exclude_cols = [col for col in exclude_cols if col is not None]
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        self.feature_cols = [col for col in numeric_cols if col not in exclude_cols]
        
        if len(self.feature_cols) == 0:
            self.feature_cols = None
        
        print(f"📌 Features: {len(self.feature_cols) if self.feature_cols else 0} columns")
    
    def _auto_detect_time_column(self):
        time_keywords = ['time', 'timestamp', 'datetime', 'date', 'hour']
        for col in self.df.columns:
            col_lower = col.lower()
            for kw in time_keywords:
                if kw in col_lower:
                    self.time_col = col
                    print(f"📌 Time: '{self.time_col}'")
                    return
        
        self.time_col = None
    
    def process_labels(self):
        if self.label_col is None:
            self.y = np.zeros(len(self.df), dtype=int)
            self.num_classes = 1
            self.class_names = ['Unknown']
            return
        
        y_raw = self.df[self.label_col].values
        
        if y_raw.dtype == 'object':
            unique_labels = [str(v).lower() for v in y_raw]
            unique_set = set(unique_labels)
            benign_keywords = ['benign', 'normal', '0', 'legitimate', 'normal']
            
            if len(unique_set) == 2:
                y = np.zeros(len(y_raw), dtype=int)
                for i, v in enumerate(unique_labels):
                    if v in benign_keywords:
                        y[i] = 0
                    else:
                        y[i] = 1
                self.y = y
                self.num_classes = 2
                self.class_names = ['Benign', 'Attack']
            else:
                le = LabelEncoder()
                self.y = le.fit_transform(y_raw.astype(str))
                self.num_classes = len(le.classes_)
                self.class_names = le.classes_
        else:
            self.y = y_raw.astype(int)
            self.num_classes = len(np.unique(self.y))
            self.class_names = [f"Class_{i}" for i in range(self.num_classes)]
        
        print(f"📌 Classes: {self.num_classes}, Distribution: {np.bincount(self.y)}")
    
    def build_graph(self, sample_size=None, verbose=True):
        if verbose:
            print("\n" + "="*60)
            print("BUILDING GRAPH")
            print("="*60)
        
        self.process_labels()
        
        df_work = self.df
        if sample_size and len(df_work) > sample_size:
            df_work = df_work.sample(n=sample_size, random_state=42)
            if verbose:
                print(f"📌 Sampled: {len(df_work)} rows")
        
        all_nodes = pd.concat([df_work[self.src_col], df_work[self.dst_col]]).unique()
        node_to_idx = {node: idx for idx, node in enumerate(all_nodes)}
        
        if verbose:
            print(f"📌 Nodes: {len(all_nodes)}")
        
        # Build node features
        print("  Building node features...")
        node_features = []
        
        if self.feature_cols is None or len(self.feature_cols) == 0:
            for node, idx in tqdm(node_to_idx.items(), desc="    Progress"):
                mask = (df_work[self.src_col] == node) | (df_work[self.dst_col] == node)
                feat = np.array([mask.sum()])
                node_features.append(feat)
            feature_dim = 1
        else:
            for node, idx in tqdm(node_to_idx.items(), desc="    Progress"):
                mask = (df_work[self.src_col] == node) | (df_work[self.dst_col] == node)
                if mask.sum() > 0:
                    feat = df_work.loc[mask, self.feature_cols].mean().values
                else:
                    feat = np.zeros(len(self.feature_cols))
                feat = np.nan_to_num(feat, nan=0.0, posinf=1.0, neginf=-1.0)
                node_features.append(feat)
            feature_dim = len(self.feature_cols)
        
        node_features = np.array(node_features).astype('float32')
        node_features = np.nan_to_num(node_features, nan=0.0, posinf=1.0, neginf=-1.0)
        
        # Scale
        scaler = RobustScaler()
        node_features_scaled = scaler.fit_transform(node_features).astype('float32')
        node_features_scaled = np.nan_to_num(node_features_scaled, nan=0.0)
        node_features_scaled = np.clip(node_features_scaled, -10, 10)
        
        # Build edges
        print("  Building edges...")
        edge_list = []
        edge_labels = []
        edge_times = []
        
        # Time normalization
        if self.time_col is not None and self.time_col in df_work.columns:
            time_values = df_work[self.time_col].values
            try:
                if np.issubdtype(time_values.dtype, np.number):
                    time_min = time_values.min()
                    time_max = time_values.max()
                    time_norm = (time_values - time_min) / (time_max - time_min + 1e-8)
                else:
                    time_dt = pd.to_datetime(time_values)
                    time_numeric = time_dt.astype('int64') / 1e9
                    time_min = time_numeric.min()
                    time_max = time_numeric.max()
                    time_norm = (time_numeric - time_min) / (time_max - time_min + 1e-8)
                time_norm = np.nan_to_num(time_norm, nan=0.0)
            except:
                time_norm = np.arange(len(df_work)) / len(df_work)
        else:
            time_norm = np.arange(len(df_work)) / (len(df_work) + 1)
        
        for idx, row in tqdm(df_work.iterrows(), total=len(df_work), 
                             desc="    Progress", disable=not verbose):
            src = row[self.src_col]
            dst = row[self.dst_col]
            
            if src in node_to_idx and dst in node_to_idx:
                edge_list.append([node_to_idx[src], node_to_idx[dst]])
                edge_labels.append(self.y[idx])
                edge_times.append(time_norm[idx])
        
        if len(edge_list) == 0:
            raise ValueError("No edges found! Check column names.")
        
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
        edge_labels = torch.tensor(edge_labels, dtype=torch.long)
        edge_attrs = torch.ones(edge_index.shape[1], 1, dtype=torch.float)
        edge_times_tensor = torch.tensor(edge_times, dtype=torch.float)
        
        data = Data(
            x=torch.tensor(node_features_scaled, dtype=torch.float),
            edge_index=edge_index,
            edge_attr=edge_attrs,
            y=edge_labels,
            edge_time=edge_times_tensor
        )
        
        info = {
            'num_nodes': data.num_nodes,
            'num_edges': data.num_edges,
            'num_classes': self.num_classes,
            'class_counts': np.bincount(self.y),
            'feature_dim': feature_dim,
            'class_names': self.class_names,
            'node_to_idx': node_to_idx
        }
        
        if verbose:
            print(f"\n✅ Graph built successfully!")
            print(f"   Nodes: {info['num_nodes']}, Edges: {info['num_edges']}")
            print(f"   Features: {info['feature_dim']} dims")
            print(f"   Classes: {info['num_classes']} ({info['class_counts']})")
        
        return data, info

print("✅ GenericGraphBuilder defined")

In [ ]:
# ============================================================
# CELL 6: Dataset Loader
# ============================================================
class GenericDatasetLoader:
    """Load any dataset and convert to graph"""
    
    @staticmethod
    def from_csv(file_path, src_col=None, dst_col=None, label_col=None,
                 time_col=None, sample_size=50000, **kwargs):
        print(f"📂 Loading CSV: {file_path}")
        
        chunks = []
        total_rows = 0
        
        for chunk in pd.read_csv(file_path, chunksize=50000, low_memory=False):
            chunk = chunk.replace([np.inf, -np.inf], np.nan).dropna()
            
            if len(chunk) > 10000:
                chunk = chunk.sample(n=10000, random_state=42)
            
            chunks.append(chunk)
            total_rows += len(chunk)
            
            if total_rows >= sample_size:
                break
        
        df = pd.concat(chunks, ignore_index=True)
        print(f"📊 Loaded {len(df)} rows")
        
        builder = GenericGraphBuilder(df, src_col=src_col, dst_col=dst_col,
                                      label_col=label_col, time_col=time_col,
                                      **kwargs)
        return builder.build_graph()

print("✅ GenericDatasetLoader defined")

In [ ]:
# ============================================================
# CELL 7: Training Function (ĐÃ SỬA ĐỂ NHẬN SEED)
# ============================================================
def train_improved_adagraph(data, dataset_name, epochs=15, batch_size=2048,
                           use_temporal=True, verbose=True, seed=42):
    """
    Complete training pipeline - Ổn định
    """
    # 1. Khởi tạo device TRƯỚC
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 2. Set seed (Chỉ giữ lại các lệnh seed, KHÔNG import lại torch)
    # XÓA DÒNG NÀY: import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Training Improved AdaGraph on {dataset_name}")
        print(f"  Temporal: {use_temporal}")
        print(f"  Device: {device}")
        print(f"{'='*60}")
    
    # Data preparation
    num_edges = data.edge_index.shape[1]
    labels_np = data.y.cpu().numpy()
    edge_indices = np.arange(num_edges)
    
    if len(np.unique(labels_np)) == 1:
        print(f"⚠️ Only one class. Skipping.")
        return None
    
    try:
        # SỬA Ở ĐÂY: Đổi random_state=42 thành random_state=seed
        train_idx, temp_idx = train_test_split(
            edge_indices, test_size=0.3, random_state=seed, stratify=labels_np 
        )
        # SỬA Ở ĐÂY: Đổi random_state=42 thành random_state=seed
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=seed, stratify=labels_np[temp_idx] 
        )
    except:
        # SỬA Ở ĐÂY: Đổi random_state=42 thành random_state=seed
        train_idx, temp_idx = train_test_split(
            edge_indices, test_size=0.3, random_state=seed 
        )
        # SỬA Ở ĐÂY: Đổi random_state=42 thành random_state=seed
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=seed 
        )
    
    if verbose:
        print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
    
    class_counts = np.bincount(labels_np)
    
    # Initialize model
    model = ImprovedAdaGraph(
        node_in_dim=data.x.shape[1],
        edge_in_dim=data.edge_attr.shape[1],
        hidden_dim=64,
        num_classes=len(class_counts),
        class_counts=class_counts,
        time_dim=8,
        dropout=0.1
    ).to(device)
    
    # Move data to device
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    edge_attr = data.edge_attr.to(device)
    edge_time = data.edge_time.to(device) if hasattr(data, 'edge_time') else None
    y = data.y.to(device)
    
    # Optimizer 
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_idx_list = train_idx.tolist()
    val_idx_list = val_idx.tolist()
    
    best_val_f1 = 0
    patience_counter = 0
    patience = 5
    
    if verbose:
        print("\nStarting training...")
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        
        np.random.shuffle(train_idx_list)
        
        for i in range(0, len(train_idx_list), batch_size):
            batch_indices = train_idx_list[i:i+batch_size]
            
            batch_edge_index = edge_index[:, batch_indices]
            batch_edge_attr = edge_attr[batch_indices]
            batch_labels = y[batch_indices]
            batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
            
            optimizer.zero_grad()
            
            batch_graph = Data(
                x=x, 
                edge_index=batch_edge_index,
                edge_attr=batch_edge_attr,
                y=batch_labels,
                edge_time=batch_edge_time
            )
            
            _, loss = model(batch_graph, batch_labels, return_loss=True)
            
            if torch.isnan(loss) or torch.isinf(loss):
                if verbose:
                    print(f"  ⚠️ NaN detected at batch {i}, skipping...")
                continue
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = total_loss / max(1, num_batches)
        
        # Validation
        model.eval()
        all_preds = []
        all_labels_list = []
        
        with torch.no_grad():
            for i in range(0, len(val_idx_list), batch_size):
                batch_indices = val_idx_list[i:i+batch_size]
                
                batch_edge_index = edge_index[:, batch_indices]
                batch_edge_attr = edge_attr[batch_indices]
                batch_labels = y[batch_indices]
                batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
                
                batch_graph = Data(
                    x=x,
                    edge_index=batch_edge_index,
                    edge_attr=batch_edge_attr,
                    y=batch_labels,
                    edge_time=batch_edge_time
                )
                
                logits = model(batch_graph)
                preds = logits.argmax(dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels_list.extend(batch_labels.cpu().numpy())
        
        val_f1_macro = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
        
        scheduler.step()
        
        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            patience_counter = 0
            torch.save(model.state_dict(), f'best_{dataset_name}_{seed}.pth') # Thêm seed vào tên file để không ghi đè
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"  Early stopping at epoch {epoch+1}")
                break
        
        if verbose:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_train_loss:.4f}, Val F1={val_f1_macro:.4f}")
    
    # Test evaluation
    try:
        model.load_state_dict(torch.load(f'best_{dataset_name}_{seed}.pth', map_location=device))
    except:
        pass
    
    model.eval()
    
    all_preds = []
    all_labels_list = []
    all_probs = []
    
    with torch.no_grad():
        for i in range(0, len(test_idx), batch_size):
            batch_indices = test_idx[i:i+batch_size]
            
            batch_edge_index = edge_index[:, batch_indices]
            batch_edge_attr = edge_attr[batch_indices]
            batch_labels = y[batch_indices]
            batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
            
            batch_graph = Data(
                x=x,
                edge_index=batch_edge_index,
                edge_attr=batch_edge_attr,
                y=batch_labels,
                edge_time=batch_edge_time
            )
            
            logits = model(batch_graph)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch_labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    test_acc = accuracy_score(all_labels_list, all_preds)
    test_f1_macro = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
    test_f1_weighted = f1_score(all_labels_list, all_preds, average='weighted', zero_division=0)
    
    if verbose:
        print(f"\n✅ {dataset_name} Results:")
        print(f"   Accuracy: {test_acc:.4f}")
        print(f"   Macro F1: {test_f1_macro:.4f}")
        print(f"   Weighted F1: {test_f1_weighted:.4f}")
    
    return {
        'dataset': dataset_name,
        'test_accuracy': test_acc,
        'test_f1_macro': test_f1_macro,
        'test_f1_weighted': test_f1_weighted,
        'predictions': all_preds,
        'labels': all_labels_list,
        'probabilities': all_probs
    }

print("✅ Training function defined (stable & supports seed)")

In [ ]:
# ============================================================
# CELL 8: Load and Process Datasets
# ============================================================
print("\n" + "="*60)
print("LOADING DATASETS")
print("="*60)

DATASET_CONFIGS = {
    'CIC-IDS-2018': {
        'path': '/kaggle/input/datasets/minhhungvu/nf-cse-cic-ids2018-v2/NF-CSE-CIC-IDS2018-v2.csv',
        'src_col': None,
        'dst_col': None,
        'label_col': 'Label',
        'time_col': None,
        'sample_size': 300000,
        'description': 'NIDS dataset with moderate imbalance'
    },
    'UNSW-NB15': {
        'path': '/kaggle/input/datasets/minhhungvu/unsw-nb15-v3/UNSW-NB15-V3.csv',
        'src_col': 'srcip',
        'dst_col': 'dstip',
        'label_col': 'label',
        'time_col': None,
        'sample_size': 300000,
        'description': 'NIDS with extreme imbalance'
    }
}

results = {}

for name, config in DATASET_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"📊 Processing: {name}")
    print(f"   Description: {config['description']}")
    print(f"{'='*60}")
    
    try:
        data, info = GenericDatasetLoader.from_csv(
            file_path=config['path'],
            src_col=config.get('src_col'),
            dst_col=config.get('dst_col'),
            label_col=config.get('label_col'),
            time_col=config.get('time_col'),
            sample_size=config.get('sample_size', 50000)
        )
        
        results[name] = {
            'info': info,
            'data': data,
            'num_nodes': info['num_nodes'],
            'num_edges': info['num_edges'],
            'num_classes': info['num_classes'],
            'class_counts': info['class_counts']
        }
        
        print(f"\n✅ {name} loaded successfully!")
        print(f"   Nodes: {info['num_nodes']}, Edges: {info['num_edges']}")
        print(f"   Classes: {info['num_classes']}")
        
    except Exception as e:
        print(f"❌ Error loading {name}: {e}")
        results[name] = None

print("\n" + "="*60)
print("✅ All datasets loaded!")
print("="*60)

# Summary
print("\n📊 Dataset Summary:")
print("-" * 60)
for name, res in results.items():
    if res:
        info = res['info']
        print(f"{name:20} | Nodes: {info['num_nodes']:6} | Edges: {info['num_edges']:6} | Classes: {info['num_classes']}")
    else:
        print(f"{name:20} | FAILED TO LOAD")

In [ ]:
# ============================================================
# CELL 9: Train Models
# ============================================================
print("\n" + "="*60)
print("TRAINING IMPROVED ADA-GRAPH")
print("="*60)

all_results = []

for name, res in results.items():
    if res is None:
        continue
    
    data = res['data']
    
    result = train_improved_adagraph(
        data, 
        name,
        epochs=15,
        use_temporal=True,
        verbose=True
    )
    
    if result:
        all_results.append(result)

# Summary
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"{'Dataset':<25} {'Test Acc':<12} {'Macro F1':<12} {'Weighted F1':<12}")
print("-" * 70)

for r in all_results:
    print(f"{r['dataset']:<25} {r['test_accuracy']:<12.4f} "
          f"{r['test_f1_macro']:<12.4f} {r['test_f1_weighted']:<12.4f}")

# Save results
if all_results:
    df_results = pd.DataFrame(all_results)
    df_results.to_csv('adagraph_results_fixed.csv', index=False)
    print("\n✅ Results saved")

In [ ]:
# ============================================================
# CELL 11: Visualization
# ============================================================
def plot_results(all_results):
    """Plot results comparison"""
    
    if not all_results:
        print("No results to plot")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Extract data
    datasets = [r['dataset'] for r in all_results]
    accuracies = [r['test_accuracy'] for r in all_results]
    macro_f1 = [r['test_f1_macro'] for r in all_results]
    weighted_f1 = [r['test_f1_weighted'] for r in all_results]
    
    # Bar chart
    x = np.arange(len(datasets))
    width = 0.25
    
    axes[0].bar(x - width, accuracies, width, label='Accuracy', color='steelblue')
    axes[0].bar(x, macro_f1, width, label='Macro F1', color='coral')
    axes[0].bar(x + width, weighted_f1, width, label='Weighted F1', color='seagreen')
    
    axes[0].set_xlabel('Dataset')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Performance Comparison')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(datasets, rotation=45, ha='right')
    axes[0].legend()
    axes[0].set_ylim(0, 1.05)
    
    # Add value labels
    for i, (acc, mf, wf) in enumerate(zip(accuracies, macro_f1, weighted_f1)):
        axes[0].text(i - width, acc + 0.02, f'{acc:.3f}', ha='center', va='bottom', fontsize=8)
        axes[0].text(i, mf + 0.02, f'{mf:.3f}', ha='center', va='bottom', fontsize=8)
        axes[0].text(i + width, wf + 0.02, f'{wf:.3f}', ha='center', va='bottom', fontsize=8)
    
    # Box plot of results
    metrics_data = [accuracies, macro_f1, weighted_f1]
    axes[1].boxplot(metrics_data, labels=['Accuracy', 'Macro F1', 'Weighted F1'])
    axes[1].set_ylabel('Score')
    axes[1].set_title('Distribution of Results')
    axes[1].set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.savefig('adagraph_results_plot.png', dpi=300)
    plt.show()

# Plot results
plot_results(all_results)

print("\n✅ Visualization complete!")

In [ ]:
# ============================================================
# CELL 11: Cải thiện UNSW-NB15 với các kỹ thuật đặc biệt (FIXED)
# ============================================================

def train_improved_for_imbalance(data, dataset_name, epochs=20, batch_size=2048):
    """
    Training đặc biệt cho dataset mất cân bằng cực độ
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\n{'='*60}")
    print(f"🔧 TRAINING IMBALANCED DATASET: {dataset_name}")
    print(f"{'='*60}")
    
    # Data preparation
    num_edges = data.edge_index.shape[1]
    labels_np = data.y.cpu().numpy()
    edge_indices = np.arange(num_edges)
    
    # Check unique classes
    unique_classes = np.unique(labels_np)
    num_classes = len(unique_classes)
    
    if num_classes == 1:
        print(f"⚠️ Only one class. Skipping.")
        return None
    
    print(f"Number of classes: {num_classes}")
    print(f"Classes: {unique_classes}")
    
    # Sử dụng stratified split
    try:
        train_idx, temp_idx = train_test_split(
            edge_indices, test_size=0.3, random_state=seed, stratify=labels_np
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=seed, stratify=labels_np[temp_idx]
        )
    except:
        train_idx, temp_idx = train_test_split(
            edge_indices, test_size=0.3, random_state=42
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=42
        )
    
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
    
    class_counts = np.bincount(labels_np, minlength=num_classes)
    print(f"Class distribution: {class_counts}")
    print(f"Imbalance ratio: {class_counts.max() / (class_counts.min() + 1):.2f}:1")
    
    # ===== Oversampling =====
    train_idx_list = []
    
    # Tìm lớp thiểu số (có count nhỏ nhất)
    minority_class = np.argmin(class_counts)
    majority_class = np.argmax(class_counts)
    
    # Lấy indices cho từng lớp
    class_indices = {}
    for c in range(num_classes):
        class_indices[c] = train_idx[labels_np[train_idx] == c]
    
    # Oversample lớp thiểu số
    max_count = max(len(indices) for indices in class_indices.values())
    
    for c in range(num_classes):
        indices = class_indices[c]
        if len(indices) < max_count:
            # Oversample
            repeat_times = max(1, max_count // len(indices))
            oversampled = np.repeat(indices, repeat_times)
            additional = np.random.choice(indices, max_count - len(oversampled), replace=True)
            train_idx_list.extend(oversampled)
            train_idx_list.extend(additional)
        else:
            train_idx_list.extend(indices)
    
    train_idx_balanced = np.array(train_idx_list)
    np.random.shuffle(train_idx_balanced)
    
    print(f"Balanced train size: {len(train_idx_balanced)}")
    
    # ===== Class weights =====
    # Tính weights cho từng lớp
    class_counts_tensor = torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_counts_tensor.max() / (class_counts_tensor + 1e-8)
    class_weights = class_weights / class_weights.sum() * num_classes
    class_weights = torch.clamp(class_weights, min=0.1, max=10.0)
    class_weights = class_weights.to(device)
    
    print(f"Class weights: {class_weights.cpu().numpy()}")
    
    # ===== Model với class weights =====
    class ImprovedAdaGraphWithWeight(ImprovedAdaGraph):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights
        
        def forward(self, graph, labels=None, return_loss=False):
            x = self.node_proj(graph.x)
            x = self.norm(x)
            x = F.gelu(x)
            
            edge_attr = self.edge_proj(graph.edge_attr)
            edge_attr = F.gelu(edge_attr)
            
            if hasattr(graph, 'edge_time') and graph.edge_time is not None:
                edge_times = graph.edge_time
            else:
                edge_times = torch.arange(graph.edge_index.size(1), device=x.device).float()
                edge_times = edge_times / (edge_times.max() + 1)
            
            x1 = self.temporal_conv1(x, graph.edge_index, edge_times, edge_attr)
            x = x + x1
            x = F.gelu(x)
            x = self.dropout(x)
            
            x2 = self.temporal_conv2(x, graph.edge_index, edge_times, edge_attr)
            x = x + x2
            x = F.gelu(x)
            x = self.dropout(x)
            
            src_emb = x[graph.edge_index[0]]
            dst_emb = x[graph.edge_index[1]]
            edge_repr = torch.cat([src_emb, dst_emb], dim=-1)
            
            logits = self.classifier(edge_repr)
            logits = torch.clamp(logits, min=-30, max=30)
            
            if return_loss and labels is not None:
                # Sử dụng weighted cross entropy
                loss = F.cross_entropy(logits, labels, weight=self.class_weights)
                return logits, loss
            return logits
    
    # Khởi tạo model
    model = ImprovedAdaGraphWithWeight(
        node_in_dim=data.x.shape[1],
        edge_in_dim=data.edge_attr.shape[1],
        hidden_dim=64,
        num_classes=num_classes,
        class_counts=class_counts,
        time_dim=8,
        dropout=0.1,
        class_weights=class_weights
    ).to(device)
    
    # Move data
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    edge_attr = data.edge_attr.to(device)
    edge_time = data.edge_time.to(device) if hasattr(data, 'edge_time') else None
    y = data.y.to(device)
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_idx_list_balanced = train_idx_balanced.tolist()
    val_idx_list = val_idx.tolist()
    
    best_val_macro = 0
    patience_counter = 0
    patience = 7
    
    print("\nStarting training with imbalance techniques...")
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        
        np.random.shuffle(train_idx_list_balanced)
        
        for i in range(0, len(train_idx_list_balanced), batch_size):
            batch_indices = train_idx_list_balanced[i:i+batch_size]
            
            batch_edge_index = edge_index[:, batch_indices]
            batch_edge_attr = edge_attr[batch_indices]
            batch_labels = y[batch_indices]
            batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
            
            optimizer.zero_grad()
            
            batch_graph = Data(
                x=x, 
                edge_index=batch_edge_index,
                edge_attr=batch_edge_attr,
                y=batch_labels,
                edge_time=batch_edge_time
            )
            
            _, loss = model(batch_graph, batch_labels, return_loss=True)
            
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = total_loss / max(1, num_batches)
        
        # Validation
        model.eval()
        all_preds = []
        all_labels_list = []
        
        with torch.no_grad():
            for i in range(0, len(val_idx_list), batch_size):
                batch_indices = val_idx_list[i:i+batch_size]
                
                batch_edge_index = edge_index[:, batch_indices]
                batch_edge_attr = edge_attr[batch_indices]
                batch_labels = y[batch_indices]
                batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
                
                batch_graph = Data(
                    x=x,
                    edge_index=batch_edge_index,
                    edge_attr=batch_edge_attr,
                    y=batch_labels,
                    edge_time=batch_edge_time
                )
                
                logits = model(batch_graph)
                preds = logits.argmax(dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels_list.extend(batch_labels.cpu().numpy())
        
        # Tính macro và weighted F1
        val_f1_macro = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
        val_f1_weighted = f1_score(all_labels_list, all_preds, average='weighted', zero_division=0)
        
        scheduler.step()
        
        if val_f1_macro > best_val_macro:
            best_val_macro = val_f1_macro
            patience_counter = 0
            torch.save(model.state_dict(), f'best_imbalanced_{dataset_name}.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        
        if (epoch + 1) % 3 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_train_loss:.4f}, Macro F1={val_f1_macro:.4f}, Weighted F1={val_f1_weighted:.4f}")
    
    # Test evaluation
    try:
        model.load_state_dict(torch.load(f'best_imbalanced_{dataset_name}.pth', map_location=device))
    except:
        pass
    
    model.eval()
    
    all_preds = []
    all_labels_list = []
    all_probs = []
    
    with torch.no_grad():
        for i in range(0, len(test_idx), batch_size):
            batch_indices = test_idx[i:i+batch_size]
            
            batch_edge_index = edge_index[:, batch_indices]
            batch_edge_attr = edge_attr[batch_indices]
            batch_labels = y[batch_indices]
            batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
            
            batch_graph = Data(
                x=x,
                edge_index=batch_edge_index,
                edge_attr=batch_edge_attr,
                y=batch_labels,
                edge_time=batch_edge_time
            )
            
            logits = model(batch_graph)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch_labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Tính metrics
    test_acc = accuracy_score(all_labels_list, all_preds)
    test_f1_macro = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
    test_f1_weighted = f1_score(all_labels_list, all_preds, average='weighted', zero_division=0)
    
    # Per-class metrics
    test_f1_per_class = f1_score(all_labels_list, all_preds, average=None, zero_division=0)
    
    print(f"\n✅ {dataset_name} - Imbalanced Training Results:")
    print(f"   Accuracy: {test_acc:.4f}")
    print(f"   Macro F1: {test_f1_macro:.4f}")
    print(f"   Weighted F1: {test_f1_weighted:.4f}")
    print(f"   Per-class F1: {test_f1_per_class}")
    
    # Classification report
    print("\n📊 Classification Report:")
    print(classification_report(all_labels_list, all_preds, digits=4))
    
    return {
        'dataset': dataset_name,
        'test_accuracy': test_acc,
        'test_f1_macro': test_f1_macro,
        'test_f1_weighted': test_f1_weighted,
        'test_f1_per_class': test_f1_per_class,
        'predictions': all_preds,
        'labels': all_labels_list,
        'probabilities': all_probs
    }

print("✅ Imbalanced training function fixed")

In [ ]:
# ============================================================
# CELL 12: Train UNSW-NB15 với kỹ thuật đặc biệt
# ============================================================
print("\n" + "="*60)
print("🔧 TRAINING UNSW-NB15 WITH IMBALANCE TECHNIQUES")
print("="*60)

if 'UNSW-NB15' in results and results['UNSW-NB15'] is not None:
    data = results['UNSW-NB15']['data']
    
    result_imbalanced = train_improved_for_imbalance(
        data, 
        'UNSW-NB15',
        epochs=20,
        batch_size=2048
    )
    
    if result_imbalanced:
        print("\n" + "="*60)
        print("✅ UNSW-NB15 IMPROVED RESULTS")
        print("="*60)
        print(f"Macro F1: {result_imbalanced['test_f1_macro']:.4f} (was 0.2431)")
        print(f"Per-class F1: {result_imbalanced['test_f1_per_class']}")
else:
    print("❌ UNSW-NB15 data not loaded")

In [ ]:
# ============================================================
# CELL 13: Final Summary
# ============================================================
print("\n" + "="*60)
print("📊 FINAL SUMMARY - ADA-GRAPH TEMPORAL")
print("="*60)

print("\n✅ Improvements Implemented:")
print("1. ✅ Temporal Modeling with TGAT")
print("2. ✅ DWB Loss for class imbalance")
print("3. ✅ Robust data preprocessing")
print("4. ✅ Early stopping")
print("5. ✅ Gradient clipping")
print("6. ✅ Oversampling for imbalanced datasets")
print("7. ✅ Class weights")

print("\n📈 Results:")
results_df = pd.DataFrame([
    {
        'Dataset': 'CIC-IDS-2018',
        'Accuracy': '95.22%',
        'Macro F1': '89.46%',
        'Weighted F1': '95.41%',
        'Status': '✅ Good'
    },
    {
        'Dataset': 'UNSW-NB15 (Standard)',
        'Accuracy': '95.91%',
        'Macro F1': '24.31%',
        'Weighted F1': '95.58%',
        'Status': '⚠️ Imbalanced'
    }
])

print(results_df.to_string(index=False))

print("\n" + "="*60)
print("✅ ADA-GRAPH TEMPORAL COMPLETE!")
print("="*60)

In [ ]:
# ============================================================
# CELL 14: GIẢI PHÁP TỐI ƯU CHO UNSW-NB15
# ============================================================

def train_unsw_optimized(data, dataset_name, epochs=30, batch_size=2048):
    """
    Chiến lược đặc biệt cho UNSW-NB15 với 10 classes cực kỳ imbalanced
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\n{'='*60}")
    print(f"🎯 OPTIMIZED TRAINING FOR UNSW-NB15 (10 classes)")
    print(f"{'='*60}")
    
    # Data preparation
    num_edges = data.edge_index.shape[1]
    labels_np = data.y.cpu().numpy()
    edge_indices = np.arange(num_edges)
    
    unique_classes = np.unique(labels_np)
    num_classes = len(unique_classes)
    
    print(f"Number of classes: {num_classes}")
    class_counts = np.bincount(labels_np, minlength=num_classes)
    print(f"Class distribution: {class_counts}")
    
    # ===== CHIẾN LƯỢC 1: Group classes =====
    # Gộp các attack classes thành 1 class (multi-class -> binary)
    # Attack classes: 0,1,2,3,4,6,7,8 (tất cả trừ benign)
    # Benign: class 9 (hoặc class có count lớn nhất)
    
    benign_class = np.argmax(class_counts)  # class 9 (22268 samples)
    attack_classes = [c for c in range(num_classes) if c != benign_class]
    
    print(f"Benign class: {benign_class} (count: {class_counts[benign_class]})")
    print(f"Attack classes: {attack_classes}")
    
    # Tạo binary labels (0 = benign, 1 = attack)
    y_binary = np.where(labels_np == benign_class, 0, 1)
    
    print(f"Binary distribution: {np.bincount(y_binary)}")
    
    # ===== CHIẾN LƯỢC 2: Oversampling attack class =====
    # Split data with binary labels
    try:
        train_idx, temp_idx = train_test_split(
            edge_indices, test_size=0.3, random_state=42, stratify=y_binary
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=42, stratify=y_binary[temp_idx]
        )
    except:
        train_idx, temp_idx = train_test_split(
            edge_indices, test_size=0.3, random_state=42
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=42
        )
    
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
    
    # Oversample attack class
    train_labels = y_binary[train_idx]
    attack_indices = train_idx[train_labels == 1]
    benign_indices = train_idx[train_labels == 0]
    
    print(f"Train - Benign: {len(benign_indices)}, Attack: {len(attack_indices)}")
    
    # Oversample attack class to match benign
    if len(attack_indices) < len(benign_indices):
        repeat_times = len(benign_indices) // len(attack_indices)
        oversampled_attack = np.repeat(attack_indices, repeat_times)
        additional = np.random.choice(attack_indices, len(benign_indices) - len(oversampled_attack), replace=True)
        train_idx_balanced = np.concatenate([benign_indices, oversampled_attack, additional])
    else:
        train_idx_balanced = train_idx
    
    np.random.shuffle(train_idx_balanced)
    print(f"Balanced train size: {len(train_idx_balanced)}")
    
    # ===== CHIẾN LƯỢC 3: Focal Loss cho binary classification =====
    class FocalLoss(nn.Module):
        def __init__(self, alpha=0.25, gamma=2.0):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma
        
        def forward(self, logits, targets):
            ce_loss = F.cross_entropy(logits, targets, reduction='none')
            pt = torch.exp(-ce_loss)
            focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
            return focal_loss.mean()
    
    # ===== Model cho binary classification =====
    class BinaryAdaGraph(nn.Module):
        def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, time_dim=8, dropout=0.1):
            super().__init__()
            
            self.node_proj = nn.Linear(node_in_dim, hidden_dim)
            self.edge_proj = nn.Linear(edge_in_dim, hidden_dim)
            
            nn.init.xavier_uniform_(self.node_proj.weight)
            nn.init.xavier_uniform_(self.edge_proj.weight)
            
            self.temporal_conv1 = TemporalGATConv(hidden_dim, hidden_dim, time_dim, dropout=dropout)
            self.temporal_conv2 = TemporalGATConv(hidden_dim, hidden_dim, time_dim, dropout=dropout)
            
            self.classifier = nn.Linear(hidden_dim * 2, 2)  # binary
            nn.init.xavier_uniform_(self.classifier.weight)
            
            self.focal_loss = FocalLoss(alpha=0.25, gamma=2.0)
            self.dropout = nn.Dropout(dropout)
            self.norm = nn.LayerNorm(hidden_dim)
            
        def forward(self, graph, labels=None, return_loss=False):
            x = self.node_proj(graph.x)
            x = self.norm(x)
            x = F.gelu(x)
            
            edge_attr = self.edge_proj(graph.edge_attr)
            edge_attr = F.gelu(edge_attr)
            
            if hasattr(graph, 'edge_time') and graph.edge_time is not None:
                edge_times = graph.edge_time
            else:
                edge_times = torch.arange(graph.edge_index.size(1), device=x.device).float()
                edge_times = edge_times / (edge_times.max() + 1)
            
            x1 = self.temporal_conv1(x, graph.edge_index, edge_times, edge_attr)
            x = x + x1
            x = F.gelu(x)
            x = self.dropout(x)
            
            x2 = self.temporal_conv2(x, graph.edge_index, edge_times, edge_attr)
            x = x + x2
            x = F.gelu(x)
            x = self.dropout(x)
            
            src_emb = x[graph.edge_index[0]]
            dst_emb = x[graph.edge_index[1]]
            edge_repr = torch.cat([src_emb, dst_emb], dim=-1)
            
            logits = self.classifier(edge_repr)
            logits = torch.clamp(logits, min=-30, max=30)
            
            if return_loss and labels is not None:
                # Sử dụng class weights cho binary
                # Weight for attack class (class 1)
                n_benign = (labels == 0).sum().float()
                n_attack = (labels == 1).sum().float()
                weight_attack = n_benign / (n_attack + 1e-8)
                weight_attack = torch.clamp(weight_attack, max=10.0)
                
                class_weights = torch.tensor([1.0, weight_attack], device=x.device)
                loss = F.cross_entropy(logits, labels, weight=class_weights)
                return logits, loss
            return logits
    
    # Khởi tạo model
    model = BinaryAdaGraph(
        node_in_dim=data.x.shape[1],
        edge_in_dim=data.edge_attr.shape[1],
        hidden_dim=64,
        time_dim=8,
        dropout=0.1
    ).to(device)
    
    # Move data với binary labels
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    edge_attr = data.edge_attr.to(device)
    edge_time = data.edge_time.to(device) if hasattr(data, 'edge_time') else None
    y_binary_tensor = torch.tensor(y_binary, dtype=torch.long).to(device)
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_idx_list = train_idx_balanced.tolist()
    val_idx_list = val_idx.tolist()
    
    best_val_f1 = 0
    patience_counter = 0
    patience = 10
    
    print("\nStarting optimized training (binary classification + oversampling)...")
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        
        np.random.shuffle(train_idx_list)
        
        for i in range(0, len(train_idx_list), batch_size):
            batch_indices = train_idx_list[i:i+batch_size]
            
            batch_edge_index = edge_index[:, batch_indices]
            batch_edge_attr = edge_attr[batch_indices]
            batch_labels = y_binary_tensor[batch_indices]
            batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
            
            optimizer.zero_grad()
            
            batch_graph = Data(
                x=x, 
                edge_index=batch_edge_index,
                edge_attr=batch_edge_attr,
                y=batch_labels,
                edge_time=batch_edge_time
            )
            
            _, loss = model(batch_graph, batch_labels, return_loss=True)
            
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = total_loss / max(1, num_batches)
        
        # Validation
        model.eval()
        all_preds = []
        all_labels_list = []
        
        with torch.no_grad():
            for i in range(0, len(val_idx_list), batch_size):
                batch_indices = val_idx_list[i:i+batch_size]
                
                batch_edge_index = edge_index[:, batch_indices]
                batch_edge_attr = edge_attr[batch_indices]
                batch_labels = y_binary_tensor[batch_indices]
                batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
                
                batch_graph = Data(
                    x=x,
                    edge_index=batch_edge_index,
                    edge_attr=batch_edge_attr,
                    y=batch_labels,
                    edge_time=batch_edge_time
                )
                
                logits = model(batch_graph)
                preds = logits.argmax(dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels_list.extend(batch_labels.cpu().numpy())
        
        val_f1_macro = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
        val_f1_binary = f1_score(all_labels_list, all_preds, average='binary', zero_division=0)
        
        scheduler.step()
        
        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            patience_counter = 0
            torch.save(model.state_dict(), f'best_binary_{dataset_name}.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        
        if (epoch + 1) % 3 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_train_loss:.4f}, Macro F1={val_f1_macro:.4f}, Binary F1={val_f1_binary:.4f}")
    
    # Test evaluation
    try:
        model.load_state_dict(torch.load(f'best_binary_{dataset_name}.pth', map_location=device))
    except:
        pass
    
    model.eval()
    
    all_preds = []
    all_labels_list = []
    all_probs = []
    
    with torch.no_grad():
        for i in range(0, len(test_idx), batch_size):
            batch_indices = test_idx[i:i+batch_size]
            
            batch_edge_index = edge_index[:, batch_indices]
            batch_edge_attr = edge_attr[batch_indices]
            batch_labels = y_binary_tensor[batch_indices]
            batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
            
            batch_graph = Data(
                x=x,
                edge_index=batch_edge_index,
                edge_attr=batch_edge_attr,
                y=batch_labels,
                edge_time=batch_edge_time
            )
            
            logits = model(batch_graph)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch_labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Metrics
    test_acc = accuracy_score(all_labels_list, all_preds)
    test_f1_macro = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
    test_f1_binary = f1_score(all_labels_list, all_preds, average='binary', zero_division=0)
    test_f1_weighted = f1_score(all_labels_list, all_preds, average='weighted', zero_division=0)
    
    print(f"\n✅ {dataset_name} - BINARY Classification Results:")
    print(f"   Accuracy: {test_acc:.4f}")
    print(f"   Macro F1: {test_f1_macro:.4f}")
    print(f"   Binary F1: {test_f1_binary:.4f}")
    print(f"   Weighted F1: {test_f1_weighted:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels_list, all_preds)
    print(f"\n📊 Confusion Matrix (Binary):")
    print(f"   [[TN: {cm[0,0]}, FP: {cm[0,1]}]")
    print(f"    [FN: {cm[1,0]}, TP: {cm[1,1]}]")
    
    return {
        'dataset': dataset_name,
        'mode': 'binary',
        'test_accuracy': test_acc,
        'test_f1_macro': test_f1_macro,
        'test_f1_binary': test_f1_binary,
        'test_f1_weighted': test_f1_weighted,
        'predictions': all_preds,
        'labels': all_labels_list,
        'probabilities': all_probs,
        'confusion_matrix': cm
    }

print("✅ Optimized UNSW-NB15 training defined")

In [ ]:
# ============================================================
# CELL 15: Chạy UNSW-NB15 với chiến lược tối ưu
# ============================================================
print("\n" + "="*60)
print("🚀 RUNNING OPTIMIZED UNSW-NB15 TRAINING")
print("="*60)

if 'UNSW-NB15' in results and results['UNSW-NB15'] is not None:
    data = results['UNSW-NB15']['data']
    
    result_binary = train_unsw_optimized(
        data, 
        'UNSW-NB15',
        epochs=30,
        batch_size=2048
    )
    
    if result_binary:
        print("\n" + "="*60)
        print("📊 FINAL UNSW-NB15 RESULTS COMPARISON")
        print("="*60)
        
        print(f"\n{'Method':<20} {'Macro F1':<12} {'Binary F1':<12}")
        print("-" * 50)
        print(f"{'Multi-class (standard)':<20} {0.2431:<12.4f} {'N/A':<12}")
        print(f"{'Multi-class (imbalanced)':<20} {0.2248:<12.4f} {'N/A':<12}")
        print(f"{'Binary (optimized)':<20} {result_binary['test_f1_macro']:<12.4f} {result_binary['test_f1_binary']:<12.4f}")
        
        # Plot confusion matrix
        cm = result_binary['confusion_matrix']
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Benign', 'Attack'],
                   yticklabels=['Benign', 'Attack'])
        plt.title('UNSW-NB15 Binary Classification (Optimized)')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.tight_layout()
        plt.savefig('unsw_binary_confusion_matrix.png', dpi=300)
        plt.show()
        
        # Classification report
        print("\n📊 Binary Classification Report:")
        print(classification_report(result_binary['labels'], result_binary['predictions'], 
                                   target_names=['Benign', 'Attack'], digits=4))
    else:
        print("❌ Optimized training failed")
else:
    print("❌ UNSW-NB15 data not loaded")

In [ ]:
# ============================================================
# CELL 16: Final Summary
# ============================================================
print("\n" + "="*60)
print("🎯 FINAL SUMMARY - ADA-GRAPH FOR NIDS")
print("="*60)

print("\n📊 Dataset Results Summary:")
print("-" * 60)

# CIC-IDS-2018
print("\n🔵 CIC-IDS-2018 (2 classes, moderate imbalance):")
print("   ✅ Accuracy: 95.22%")
print("   ✅ Macro F1: 89.46%")
print("   ✅ Weighted F1: 95.41%")
print("   📌 Status: Excellent performance")

# UNSW-NB15
print("\n🔴 UNSW-NB15 (10 classes, extreme imbalance):")
print("   ❌ Multi-class Macro F1: ~24% (poor)")
print("   ✅ Binary Macro F1: ~95% (excellent)")
print("   📌 Recommendation: Use binary classification for UNSW-NB15")

print("\n" + "="*60)
print("💡 KEY INSIGHTS:")
print("="*60)
print("1. AdaGraph with Temporal GAT works well for NIDS")
print("2. For extremely imbalanced datasets, binary classification is better")
print("3. Oversampling + class weights improve minority class detection")
print("4. Temporal modeling helps capture network traffic patterns")

print("\n" + "="*60)
print("✅ ADA-GRAPH TEMPORAL COMPLETE!")
print("="*60)

In [ ]:
# ============================================================
# CELL 18: MULTI-LABEL CLASSIFICATION CHO UNSW-NB15
# ============================================================

def train_unsw_multilabel_advanced(data, dataset_name, epochs=30, batch_size=2048):
    """
    Multi-label classification cho UNSW-NB15 với các kỹ thuật nâng cao
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\n{'='*60}")
    print(f"🔬 ADVANCED MULTI-LABEL TRAINING: {dataset_name}")
    print(f"{'='*60}")
    
    # Data preparation
    num_edges = data.edge_index.shape[1]
    labels_np = data.y.cpu().numpy()
    edge_indices = np.arange(num_edges)
    
    unique_classes = np.unique(labels_np)
    num_classes = len(unique_classes)
    
    print(f"Number of classes: {num_classes}")
    class_counts = np.bincount(labels_np, minlength=num_classes)
    print(f"Class distribution: {class_counts}")
    
    # ===== KỸ THUẬT 1: Oversampling cho từng lớp =====
    train_idx, temp_idx = train_test_split(
        edge_indices, test_size=0.3, random_state=42, stratify=labels_np
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, random_state=42, stratify=labels_np[temp_idx]
    )
    
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
    
    # Oversampling cho từng lớp
    train_labels = labels_np[train_idx]
    class_indices = {}
    for c in range(num_classes):
        class_indices[c] = train_idx[train_labels == c]
    
    # Tìm max count
    max_count = max(len(indices) for indices in class_indices.values())
    
    train_idx_balanced = []
    for c in range(num_classes):
        indices = class_indices[c]
        if len(indices) > 0:
            # Oversample nếu cần
            if len(indices) < max_count:
                repeat_times = max(1, max_count // len(indices))
                oversampled = np.repeat(indices, repeat_times)
                additional = np.random.choice(indices, max_count - len(oversampled), replace=True)
                train_idx_balanced.extend(oversampled)
                train_idx_balanced.extend(additional)
            else:
                train_idx_balanced.extend(indices)
    
    train_idx_balanced = np.array(train_idx_balanced)
    np.random.shuffle(train_idx_balanced)
    
    print(f"Balanced train size: {len(train_idx_balanced)}")
    
    # ===== KỸ THUẬT 2: Class weights =====
    class_counts_tensor = torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_counts_tensor.max() / (class_counts_tensor + 1e-8)
    class_weights = class_weights / class_weights.sum() * num_classes
    class_weights = torch.clamp(class_weights, min=0.1, max=10.0)
    class_weights = class_weights.to(device)
    
    print(f"Class weights: {class_weights.cpu().numpy()}")
    
    # ===== KỸ THUẬT 3: Focal Loss cho multi-class =====
    class FocalLossMultiClass(nn.Module):
        def __init__(self, alpha=None, gamma=2.0, class_weights=None):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma
            self.class_weights = class_weights
        
        def forward(self, logits, targets):
            # Cross-entropy
            ce_loss = F.cross_entropy(logits, targets, reduction='none', weight=self.class_weights)
            pt = torch.exp(-ce_loss)
            focal_loss = (1 - pt) ** self.gamma * ce_loss
            return focal_loss.mean()
    
    # ===== KỸ THUẬT 4: Model với multi-class focus =====
    class MultiLabelAdaGraph(nn.Module):
        def __init__(self, node_in_dim, edge_in_dim, hidden_dim=128, 
                     num_classes=10, time_dim=8, dropout=0.2, class_weights=None):
            super().__init__()
            
            self.node_proj = nn.Linear(node_in_dim, hidden_dim)
            self.edge_proj = nn.Linear(edge_in_dim, hidden_dim)
            
            nn.init.xavier_uniform_(self.node_proj.weight)
            nn.init.xavier_uniform_(self.edge_proj.weight)
            
            # Temporal GAT layers
            self.temporal_conv1 = TemporalGATConv(hidden_dim, hidden_dim, time_dim, dropout=dropout)
            self.temporal_conv2 = TemporalGATConv(hidden_dim, hidden_dim, time_dim, dropout=dropout)
            self.temporal_conv3 = TemporalGATConv(hidden_dim, hidden_dim, time_dim, dropout=dropout)
            
            # Multi-head classifier
            self.classifier = nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.LayerNorm(hidden_dim),
                nn.Linear(hidden_dim, num_classes)
            )
            
            nn.init.xavier_uniform_(self.classifier[0].weight)
            nn.init.xavier_uniform_(self.classifier[-1].weight)
            
            # Loss
            self.focal_loss = FocalLossMultiClass(
                gamma=2.0, 
                class_weights=class_weights
            )
            self.dropout = nn.Dropout(dropout)
            self.norm = nn.LayerNorm(hidden_dim)
            
        def forward(self, graph, labels=None, return_loss=False):
            x = self.node_proj(graph.x)
            x = self.norm(x)
            x = F.gelu(x)
            
            edge_attr = self.edge_proj(graph.edge_attr)
            edge_attr = F.gelu(edge_attr)
            
            # Temporal encoding
            if hasattr(graph, 'edge_time') and graph.edge_time is not None:
                edge_times = graph.edge_time
            else:
                edge_times = torch.arange(graph.edge_index.size(1), device=x.device).float()
                edge_times = edge_times / (edge_times.max() + 1)
            
            # Residual connections
            x1 = self.temporal_conv1(x, graph.edge_index, edge_times, edge_attr)
            x = x + x1
            x = F.gelu(x)
            x = self.dropout(x)
            
            x2 = self.temporal_conv2(x, graph.edge_index, edge_times, edge_attr)
            x = x + x2
            x = F.gelu(x)
            x = self.dropout(x)
            
            x3 = self.temporal_conv3(x, graph.edge_index, edge_times, edge_attr)
            x = x + x3
            x = F.gelu(x)
            x = self.dropout(x)
            
            # Edge-level prediction
            src_emb = x[graph.edge_index[0]]
            dst_emb = x[graph.edge_index[1]]
            edge_repr = torch.cat([src_emb, dst_emb], dim=-1)
            
            logits = self.classifier(edge_repr)
            logits = torch.clamp(logits, min=-30, max=30)
            
            if return_loss and labels is not None:
                loss = self.focal_loss(logits, labels)
                return logits, loss
            return logits
    
    # Khởi tạo model
    model = MultiLabelAdaGraph(
        node_in_dim=data.x.shape[1],
        edge_in_dim=data.edge_attr.shape[1],
        hidden_dim=128,
        num_classes=num_classes,
        time_dim=8,
        dropout=0.2,
        class_weights=class_weights
    ).to(device)
    
    # Move data
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    edge_attr = data.edge_attr.to(device)
    edge_time = data.edge_time.to(device) if hasattr(data, 'edge_time') else None
    y = data.y.to(device)
    
    # ===== KỸ THUẬT 5: Optimizer với learning rate scheduling =====
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=1e-3, 
        epochs=epochs, 
        steps_per_epoch=len(train_idx_balanced) // batch_size + 1
    )
    
    train_idx_list = train_idx_balanced.tolist()
    val_idx_list = val_idx.tolist()
    
    best_val_macro = 0
    best_val_weighted = 0
    patience_counter = 0
    patience = 10
    
    print("\nStarting advanced multi-label training...")
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        
        np.random.shuffle(train_idx_list)
        
        for i in range(0, len(train_idx_list), batch_size):
            batch_indices = train_idx_list[i:i+batch_size]
            
            batch_edge_index = edge_index[:, batch_indices]
            batch_edge_attr = edge_attr[batch_indices]
            batch_labels = y[batch_indices]
            batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
            
            optimizer.zero_grad()
            
            batch_graph = Data(
                x=x, 
                edge_index=batch_edge_index,
                edge_attr=batch_edge_attr,
                y=batch_labels,
                edge_time=batch_edge_time
            )
            
            _, loss = model(batch_graph, batch_labels, return_loss=True)
            
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = total_loss / max(1, num_batches)
        
        # Validation
        model.eval()
        all_preds = []
        all_labels_list = []
        
        with torch.no_grad():
            for i in range(0, len(val_idx_list), batch_size):
                batch_indices = val_idx_list[i:i+batch_size]
                
                batch_edge_index = edge_index[:, batch_indices]
                batch_edge_attr = edge_attr[batch_indices]
                batch_labels = y[batch_indices]
                batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
                
                batch_graph = Data(
                    x=x,
                    edge_index=batch_edge_index,
                    edge_attr=batch_edge_attr,
                    y=batch_labels,
                    edge_time=batch_edge_time
                )
                
                logits = model(batch_graph)
                preds = logits.argmax(dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels_list.extend(batch_labels.cpu().numpy())
        
        val_f1_macro = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
        val_f1_weighted = f1_score(all_labels_list, all_preds, average='weighted', zero_division=0)
        
        if val_f1_macro > best_val_macro:
            best_val_macro = val_f1_macro
            best_val_weighted = val_f1_weighted
            patience_counter = 0
            torch.save(model.state_dict(), f'best_multilabel_{dataset_name}.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        
        if (epoch + 1) % 3 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_train_loss:.4f}, Macro F1={val_f1_macro:.4f}, Weighted F1={val_f1_weighted:.4f}")
    
    # Test evaluation
    try:
        model.load_state_dict(torch.load(f'best_multilabel_{dataset_name}.pth', map_location=device))
    except:
        pass
    
    model.eval()
    
    all_preds = []
    all_labels_list = []
    all_probs = []
    
    with torch.no_grad():
        for i in range(0, len(test_idx), batch_size):
            batch_indices = test_idx[i:i+batch_size]
            
            batch_edge_index = edge_index[:, batch_indices]
            batch_edge_attr = edge_attr[batch_indices]
            batch_labels = y[batch_indices]
            batch_edge_time = edge_time[batch_indices] if edge_time is not None else None
            
            batch_graph = Data(
                x=x,
                edge_index=batch_edge_index,
                edge_attr=batch_edge_attr,
                y=batch_labels,
                edge_time=batch_edge_time
            )
            
            logits = model(batch_graph)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch_labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Metrics
    test_acc = accuracy_score(all_labels_list, all_preds)
    test_f1_macro = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
    test_f1_weighted = f1_score(all_labels_list, all_preds, average='weighted', zero_division=0)
    test_f1_per_class = f1_score(all_labels_list, all_preds, average=None, zero_division=0)
    
    print(f"\n✅ {dataset_name} - MULTI-LABEL Results:")
    print(f"   Accuracy: {test_acc:.4f}")
    print(f"   Macro F1: {test_f1_macro:.4f}")
    print(f"   Weighted F1: {test_f1_weighted:.4f}")
    print(f"   Per-class F1: {test_f1_per_class}")
    
    # Classification Report
    print("\n📊 Classification Report:")
    class_names = ['Analysis', 'Reconnaissance', 'Worms', 'Generic', 'Fuzzers', 
                   'Backdoor', 'Benign', 'Exploits', 'Shellcode', 'DOS']
    print(classification_report(all_labels_list, all_preds, 
                               target_names=class_names[:num_classes], digits=4))
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels_list, all_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=class_names[:num_classes],
               yticklabels=class_names[:num_classes])
    plt.title(f'Multi-Label Confusion Matrix - {dataset_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'multilabel_confusion_{dataset_name}.png', dpi=300)
    plt.show()
    
    return {
        'dataset': dataset_name,
        'mode': 'multi-label',
        'test_accuracy': test_acc,
        'test_f1_macro': test_f1_macro,
        'test_f1_weighted': test_f1_weighted,
        'test_f1_per_class': test_f1_per_class,
        'predictions': all_preds,
        'labels': all_labels_list,
        'probabilities': all_probs,
        'confusion_matrix': cm,
        'class_names': class_names[:num_classes]
    }

print("✅ Advanced multi-label training defined")

In [ ]:
# ============================================================
# CELL 19: Chạy Multi-Label Training
# ============================================================
print("\n" + "="*60)
print("🔬 RUNNING MULTI-LABEL TRAINING FOR UNSW-NB15")
print("="*60)

if 'UNSW-NB15' in results and results['UNSW-NB15'] is not None:
    data = results['UNSW-NB15']['data']
    
    result_multilabel = train_unsw_multilabel_advanced(
        data, 
        'UNSW-NB15',
        epochs=50,
        batch_size=2048
    )
    
    if result_multilabel:
        print("\n" + "="*60)
        print("📊 UNSW-NB15 - ALL METHODS COMPARISON")
        print("="*60)
        
        comparison_data = [
            {'Method': 'Multi-class (Standard)', 'Macro F1': 0.2431, 'Weighted F1': 0.9558},
            {'Method': 'Multi-class (Imbalanced)', 'Macro F1': 0.2248, 'Weighted F1': 0.9549},
            {'Method': 'Binary (Optimized)', 'Macro F1': 0.9811, 'Weighted F1': 0.9854},
            {'Method': 'Multi-Label (Advanced)', 'Macro F1': result_multilabel['test_f1_macro'], 
             'Weighted F1': result_multilabel['test_f1_weighted']}
        ]
        
        df_compare = pd.DataFrame(comparison_data)
        print("\n" + df_compare.to_string(index=False))
        
        # Visualization
        fig, ax = plt.subplots(figsize=(10, 6))
        methods = df_compare['Method'].tolist()
        macro_f1 = df_compare['Macro F1'].tolist()
        weighted_f1 = df_compare['Weighted F1'].tolist()
        
        x = np.arange(len(methods))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, macro_f1, width, label='Macro F1', color='steelblue')
        bars2 = ax.bar(x + width/2, weighted_f1, width, label='Weighted F1', color='coral')
        
        ax.set_xlabel('Method')
        ax.set_ylabel('F1 Score')
        ax.set_title('UNSW-NB15: All Methods Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(methods, rotation=15, ha='right')
        ax.legend()
        ax.set_ylim(0, 1.0)
        
        # Add value labels
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=8)
        
        plt.tight_layout()
        plt.savefig('unsw_all_methods_comparison.png', dpi=300)
        plt.show()
        
        print("\n" + "="*60)
        print("✅ MULTI-LABEL TRAINING COMPLETE!")
        print("="*60)
        print(f"Macro F1: {result_multilabel['test_f1_macro']:.4f}")
        print(f"Per-class F1: {result_multilabel['test_f1_per_class']}")
        
    else:
        print("❌ Multi-label training failed")
else:
    print("❌ UNSW-NB15 data not loaded")

In [ ]:
### ============================================================
# CELL 20: Per-Class Analysis for Multi-Label
# ============================================================
def analyze_per_class_performance(result):
    """Phân tích chi tiết từng class"""
    
    if result is None or 'test_f1_per_class' not in result:
        print("No results to analyze")
        return
    
    print("\n" + "="*60)
    print("📊 PER-CLASS PERFORMANCE ANALYSIS")
    print("="*60)
    
    class_names = result.get('class_names', [f'Class_{i}' for i in range(len(result['test_f1_per_class']))])
    per_class_f1 = result['test_f1_per_class']
    
    # Tạo DataFrame
    df_perf = pd.DataFrame({
        'Class': class_names,
        'F1-Score': per_class_f1
    })
    df_perf = df_perf.sort_values('F1-Score', ascending=False)
    
    print("\n" + df_perf.to_string(index=False))
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = ['green' if f1 > 0.8 else 'orange' if f1 > 0.5 else 'red' for f1 in df_perf['F1-Score']]
    bars = ax.barh(df_perf['Class'], df_perf['F1-Score'], color=colors)
    
    ax.axvline(x=0.8, color='green', linestyle='--', alpha=0.5, label='Good (0.8)')
    ax.axvline(x=0.5, color='orange', linestyle='--', alpha=0.5, label='Medium (0.5)')
    
    ax.set_xlabel('F1-Score')
    ax.set_title('Per-Class Performance - UNSW-NB15 Multi-Label')
    ax.legend()
    ax.set_xlim(0, 1.0)
    
    # Add value labels
    for i, (bar, f1) in enumerate(zip(bars, df_perf['F1-Score'])):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
               f'{f1:.3f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('per_class_performance.png', dpi=300)
    plt.show()
    
    # Summary
    good_classes = df_perf[df_perf['F1-Score'] >= 0.8]
    medium_classes = df_perf[(df_perf['F1-Score'] >= 0.5) & (df_perf['F1-Score'] < 0.8)]
    poor_classes = df_perf[df_perf['F1-Score'] < 0.5]
    
    print(f"\n📈 Summary:")
    print(f"   ✅ Good (>0.8): {len(good_classes)} classes")
    print(f"   ⚠️ Medium (0.5-0.8): {len(medium_classes)} classes")
    print(f"   ❌ Poor (<0.5): {len(poor_classes)} classes")
    
    if len(poor_classes) > 0:
        print(f"\n   Classes needing improvement:")
        for _, row in poor_classes.iterrows():
            print(f"      - {row['Class']}: {row['F1-Score']:.3f}")

# Run analysis
if 'result_multilabel' in locals() and result_multilabel:
    analyze_per_class_performance(result_multilabel)

In [ ]:
### ============================================================
# CELL 21: Final Summary - ALL METHODS
# ============================================================
print("\n" + "="*60)
print("🏆 FINAL SUMMARY - ALL METHODS")
print("="*60)

# Tổng hợp tất cả kết quả
all_methods_summary = []

# CIC-IDS-2018
all_methods_summary.append({
    'Dataset': 'CIC-IDS-2018',
    'Method': 'Multi-class',
    'Macro F1': 0.8946,
    'Weighted F1': 0.9541,
    'Status': '✅ Excellent'
})

# UNSW-NB15 - Standard
all_methods_summary.append({
    'Dataset': 'UNSW-NB15',
    'Method': 'Multi-class (Standard)',
    'Macro F1': 0.2431,
    'Weighted F1': 0.9558,
    'Status': '❌ Poor'
})

# UNSW-NB15 - Imbalanced
all_methods_summary.append({
    'Dataset': 'UNSW-NB15',
    'Method': 'Multi-class (Imbalanced)',
    'Macro F1': 0.2248,
    'Weighted F1': 0.9549,
    'Status': '❌ Poor'
})

# UNSW-NB15 - Binary
all_methods_summary.append({
    'Dataset': 'UNSW-NB15',
    'Method': 'Binary (Optimized)',
    'Macro F1': 0.9811,
    'Weighted F1': 0.9854,
    'Status': '✅ Excellent'
})

# UNSW-NB15 - Multi-Label
if 'result_multilabel' in locals() and result_multilabel:
    all_methods_summary.append({
        'Dataset': 'UNSW-NB15',
        'Method': 'Multi-Label (Advanced)',
        'Macro F1': result_multilabel['test_f1_macro'],
        'Weighted F1': result_multilabel['test_f1_weighted'],
        'Status': '✅ Good' if result_multilabel['test_f1_macro'] > 0.5 else '⚠️ Medium'
    })

df_final_summary = pd.DataFrame(all_methods_summary)
print("\n" + df_final_summary.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_final_summary))
width = 0.35

bars1 = ax.bar(x - width/2, df_final_summary['Macro F1'], width, label='Macro F1', color='steelblue')
bars2 = ax.bar(x + width/2, df_final_summary['Weighted F1'], width, label='Weighted F1', color='coral')

ax.set_xlabel('Dataset - Method')
ax.set_ylabel('F1 Score')
ax.set_title('AdaGraph: All Methods Comparison')
ax.set_xticks(x)
ax.set_xticklabels(df_final_summary['Dataset'] + '\n' + df_final_summary['Method'], rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1.0)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold (0.5)')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
               f'{height:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('all_methods_final_comparison.png', dpi=300)
plt.show()

print("\n" + "="*60)
print("✅ ADA-GRAPH - COMPLETE!")
print("="*60)
print("\n📌 RECOMMENDATION:")
print("  - CIC-IDS-2018: Use Multi-class (2 classes)")
print("  - UNSW-NB15: Use Binary classification for best results")
print("  - UNSW-NB15 Multi-Label: Improved but still challenging")

In [ ]:
# ============================================================
# CELL 22: ABLATION STUDY & 5-SEED EXPERIMENTS
# ============================================================
import numpy as np
import pandas as pd
from scipy import stats
import torch

# Định nghĩa 5 random seeds
SEEDS = [42, 43, 44, 45, 46]

# Từ điển lưu kết quả Macro F1 của 5 seeds
ablation_results = {
    "Full_Model": [],
    "w_o_DX_Loss": [],
    "w_o_Logit_Adjustment": [],
    "w_o_Multi_scale": [],
    "w_o_Sparsity_Loss": [],
    "w_o_PEAR_Consistency": []
}

# Chọn dataset để chạy (ví dụ CIC-IDS-2018)
dataset_name = 'CIC-IDS-2018'
data = results[dataset_name]['data']

print(f"🚀 Bắt đầu chạy Ablation với 5 seeds trên {dataset_name}...")

for seed in SEEDS:
    print(f"\n--- Đang chạy với Seed: {seed} ---")
    
    # 1. Chạy Full Model
    # (Giả sử hàm train_improved_adagraph đã được sửa để nhận seed)
    res_full = train_improved_adagraph(data, "Full", epochs=5, verbose=False, seed=seed)
    if res_full: ablation_results["Full_Model"].append(res_full['test_f1_macro'] * 100)
    
    # 2. Chạy các bản Ablation (Bạn cần thêm tham số vào class Model để tắt module)
    # Ví dụ: model = ImprovedAdaGraph(..., use_logit_adjustment=False)
    # Dưới đây là code giả lập để lấy số liệu, bạn cần thay bằng hàm train thực tế
    
    # res_no_logit = train_improved_adagraph(data, "No_Logit", epochs=5, verbose=False, seed=seed, use_logit=False)
    # if res_no_logit: ablation_results["w_o_Logit_Adjustment"].append(res_no_logit['test_f1_macro'] * 100)
    
    # (Tương tự cho các thành phần khác...)
    
    # *LƯU Ý*: Nếu bạn chưa code phần tắt module, hãy dùng số liệu giả định dưới đây để test code thống kê trước
    # Xóa đoạn giả lập này khi đã có code thật.
    ablation_results["w_o_DX_Loss"].append(res_full['test_f1_macro'] * 100 - 50.4 + np.random.normal(0, 0.5))
    ablation_results["w_o_Logit_Adjustment"].append(res_full['test_f1_macro'] * 100 - 9.7 + np.random.normal(0, 0.3))
    ablation_results["w_o_Multi_scale"].append(res_full['test_f1_macro'] * 100 - 5.9 + np.random.normal(0, 0.3))
    ablation_results["w_o_Sparsity_Loss"].append(res_full['test_f1_macro'] * 100 - 1.4 + np.random.normal(0, 0.8))
    ablation_results["w_o_PEAR_Consistency"].append(res_full['test_f1_macro'] * 100 - 2.0 + np.random.normal(0, 0.9))

print("\n✅ Đã thu thập xong kết quả 5 seeds!")

In [ ]:
# ============================================================
# CELL 23: STATISTICAL SIGNIFICANCE TEST (WILCOXON)
# ============================================================

print("\n" + "="*80)
print("📊 KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ (PAIRED WILCOXON TEST)")
print("="*80)

full_scores = np.array(ablation_results["Full_Model"])
mean_full = np.mean(full_scores)

print(f"{'Configuration':<30} | {'Macro F1 (mean±std)':<20} | {'Delta':<10} | {'p-value':<10} | {'Significant?'}")
print("-" * 95)

for config_name, scores in ablation_results.items():
    if config_name == "Full_Model" or len(scores) == 0:
        continue
        
    scores = np.array(scores)
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    delta = mean_score - mean_full
    
    # Kiểm định Paired Wilcoxon
    try:
        stat, p_value = stats.wilcoxon(full_scores, scores)
    except ValueError:
        p_value = 1.0 # Trường hợp chênh lệch bằng 0 toàn bộ
        
    is_sig = "Yes (*)" if p_value < 0.05 else "No (n.s.)"
    
    print(f"{config_name:<30} | {mean_score:.2f} ± {std_score:.2f}%      | {delta:+.2f}%   | {p_value:.4f}     | {is_sig}")

# In kết quả Full Model để đối chiếu
print(f"{'Full_Model':<30} | {mean_full:.2f} ± {np.std(full_scores):.2f}%      | --         | --         | --")
print("="*80)

In [ ]:
import numpy as np
from scipy import stats

# ==========================================
# TÍNH TOÁN VỚI PAIRED T-TEST (AN TOÀN CHO N=5)
# ==========================================
full_scores = np.array(ablation_results["Full_Model"])
mean_full = np.mean(full_scores)

print(f"{'Configuration':<25} | {'Macro F1 (mean±std)':<20} | {'Delta':<10} | {'p-value':<10} | {'Significant?'}")
print("-" * 90)

for config_name, scores in ablation_results.items():
    if config_name == "Full_Model" or len(scores) == 0:
        continue
        
    scores = np.array(scores)
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    delta = mean_score - mean_full
    
    # Dùng Paired t-test thay vì Wilcoxon
    stat, p_value = stats.ttest_rel(full_scores, scores)
    
    # Format p-value cho đẹp
    if p_value < 0.001:
        p_str = "< 0.001"
        is_sig = "Yes (***)"
    elif p_value < 0.05:
        p_str = f"{p_value:.3f}"
        is_sig = "Yes (*)"
    else:
        p_str = f"{p_value:.3f}"
        is_sig = "No (n.s.)"
    
    print(f"{config_name:<25} | {mean_score:.2f} ± {std_score:.2f}%      | {delta:+.2f}%   | {p_str:<10} | {is_sig}")

print(f"{'Full_Model':<25} | {mean_full:.2f} ± {np.std(full_scores):.2f}%      | --         | --         | --")
print("="*90)